# 01｜Swin Transformer 概述与四阶段结构

前面已经学习了 CNN、Attention、Transformer Encoder 和 ViT。现在开始学习 Swin Transformer。

这一课只建立整体认识，不写模型代码，也不提前进入训练。重点是理解 Swin 为什么出现，以及它相对普通 ViT 改变了什么。

![Swin-T 四阶段整体结构](images/01_swin_stages.svg)

## 1. 为什么在 ViT 之后学习 Swin

ViT 把图片切成 patch tokens，再使用 Self-Attention 让所有 tokens 交换信息。它把 Transformer 成功应用到了图像任务中。

但普通 ViT 的全局 Self-Attention 有一个明显问题：每个 token 都要和所有 token 计算关系。图片越大，token 越多，注意力矩阵增长得越快。

例如，$224\times224$ 的图片使用 $4\times4$ patch 时，每边有 56 个 patch，总共有：

$$
N=56\times56=3136
$$

全局 Attention 需要处理一个 $3136\times3136$ 的关系矩阵。以后处理更高分辨率的图片时，计算和显存压力还会继续增加。

Swin Transformer 的出发点，就是在保留 Attention 建模能力的同时，让视觉 Transformer 更适合高分辨率和多尺度图像任务。

## 补充：为什么算力/显存压力会连累大图里的小物体识别


   - 自注意力是序列长度 N 的二次方复杂度，而 N 随分辨率线性增长：

     大图 → patch 变多 → 序列变长 → 计算和显存按 N² 暴涨，很快爆显存。

   - 为省资源只能二选一：① 下采样大图（分辨率降低，小物体本就几个像素，再压缩就只剩 1 个甚至不到 1 个 patch，信息基本丢失）；② 增大 patch 尺寸（等效降低有效分辨率，小物体细节被抹平）。
   - 结果：小物体最终只对应极少数 token，在全局注意力里被淹没，难以保留足够的局部细节，导致漏检/误识别。
   - 一句话：算力限制逼你用粗粒度特征表示大图，而小物体恰恰需要细粒度的局部特征。

## 2. Swin Transformer 是什么

Swin Transformer 是一种具有局部窗口和层级结构的视觉 Transformer。

名字中的 Swin 来自 **Shifted Window**，也就是“移动窗口”。它的核心是让相邻层使用不同的窗口划分方式，使原本位于不同窗口的 tokens 也能逐层交换信息。

可以先记住四个关键词：

1. Window Attention：只在局部窗口内计算注意力。
2. Shifted Window：移动下一层的窗口边界。
3. Patch Merging：合并相邻 tokens，降低空间分辨率。
4. Hierarchical Structure：形成由浅到深的多阶段层级特征。

## 3. ViT 与 Swin 的主要区别

| 对比角度 | 普通 ViT | Swin Transformer |
|---|---|---|
| Attention 范围 | 所有 tokens 全局交互 | 先在局部窗口内交互 |
| token 分辨率 | Encoder 中通常保持不变 | 随 Stage 逐步降低 |
| 特征维度 | 通常保持不变 | 随 Stage 逐步增加 |
| 特征结构 | 单一尺度为主 | 多阶段层级结构 |
| 窗口间通信 | 全局 Attention 天然支持 | 依靠 Shifted Window |
| 与 CNN 的相似点 | 相对较少 | 局部计算、下采样、层级特征 |

Swin 仍然属于 Transformer。它的 Block 中仍然包含 Attention、FFN、残差连接和 LayerNorm，只是改变了 Attention 的作用范围，并加入了层级下采样。

## 4. Window Attention：在局部窗口内计算

假设当前特征图的空间大小（tokens 数量）是 $56\times56$，窗口大小是 $7\times7$。

每个窗口包含：

$$
7\times7=49\text{ 个 tokens}
$$

整张特征图一共可以划分成：

$$
\frac{56}{7}\times\frac{56}{7}=8\times8=64\text{ 个窗口}
$$

每个窗口独立进行 Self-Attention。一个 token 只与同一窗口中的 tokens 直接计算关系，不会在这一层立即查看整张图片。

这与 CNN 的局部感受野有相似之处：两者都先关注局部区域。但 CNN 使用卷积核进行固定形式的局部计算，而 Swin 在窗口内使用 Q、K、V 动态计算注意力权重。

## 5. Window Attention 为什么更省计算

设整张特征图共有 $N$ 个 tokens，每个窗口包含 $M^2$ 个 tokens。

全局 Self-Attention 的注意力关系数量随 $N^2$ 增长，因为每个 token 都要查看所有 tokens。

Window Attention 把关系计算限制在固定大小的窗口内。窗口数量会随图片大小增加，但单个窗口中的 token 数量保持不变。因此，当窗口大小固定时，注意力部分的计算量会更接近随 token 数量线性增长。

最重要的直觉是：

> 全局 Attention 扩大图片时，会同时扩大每个 token 的查找范围；Window Attention 扩大图片时，主要是增加窗口数量，而每个窗口的计算规模仍然受限制。

## 6. 固定窗口带来的问题

只使用 Window Attention 还不够。

如果每一层都使用完全相同的窗口划分，那么窗口内部可以交换信息，但不同窗口之间始终被边界隔开。位于两个相邻窗口中的 tokens，即使空间位置很接近，也不能直接交流。

这样会让图片被分成许多互不通信的小区域，限制模型理解跨区域结构的能力。

因此，Window Attention 解决了计算量问题，但同时引出了窗口之间怎样通信的问题。

## 7. Shifted Window：让窗口之间交换信息

Swin 通常让两个连续 Block 使用不同的窗口划分：

- 第一个 Block 使用规则窗口，称为 W-MSA。
- 第二个 Block 把窗口边界移动约半个窗口，称为 SW-MSA。

窗口边界移动后，原来位于不同窗口中的一部分 tokens 会进入同一个新窗口。它们便可以在第二个 Block 中交换信息。

Shifted Window 移动的不是图片内容，也不是 token 本身，而是这一层对 tokens 的分组方式。

连续多层交替使用规则窗口和移动窗口后，信息就能逐步跨越原来的窗口边界，同时每一层仍只需要进行局部 Attention。

## 8. Patch Merging：降低分辨率并增加通道

Swin 不会从头到尾保持相同数量的 tokens。不同 Stage 之间使用 Patch Merging 进行下采样。

Patch Merging 会取相邻的 $2\times2$ 个 tokens，把它们的特征拼接起来，再使用线性映射进行压缩。

形状变化可以概括为：

$$
B\times H\times W\times C
\longrightarrow
B\times\frac{H}{2}\times\frac{W}{2}\times2C
$$

变化结果是：

- 高和宽各变为原来的一半；
- token 数量变为原来的四分之一；
- 通道数通常变为原来的两倍；
- 每个新 token 表示更大的图片区域。

它与 CNN 中的池化或步长卷积作用相似：空间尺寸逐步降低，特征语义逐步增强。

## 9. Swin-T 的四阶段结构

Swin-T 接收 $224\times224$ RGB 图片时，主要形状变化如下：

| 位置 | 空间分辨率 | 通道数 | 含义 |
|---|---:|---:|---|
| 输入图片 | $224\times224$ | 3 | 原始 RGB 图片 |
| Patch Embedding | $56\times56$ | 96 | 每个 token 对应 $4\times4$ 图片区域 |
| Stage 1 | $56\times56$ | 96 | 保持当前分辨率，更新局部特征 |
| Stage 2 | $28\times28$ | 192 | 第一次 Patch Merging 后 |
| Stage 3 | $14\times14$ | 384 | 第二次 Patch Merging 后 |
| Stage 4 | $7\times7$ | 768 | 第三次 Patch Merging 后 |
| 分类表示 | $1\times1$ | 768 | 对 $7\times7$ 位置做全局平均池化 |

这条路线与 CNN 很相似：分辨率逐层降低，通道逐层增加。浅层保留更多位置细节，深层表示更大的区域和更强的语义信息。

Swin-T 中的 T 表示 Tiny 规模配置，不是我们之前自己实现的 TinyViT。

## 10. Swin 与已经学过的知识怎样连接

### 与 Attention 的连接

窗口内部仍然使用 Q、K、V、Softmax 和加权求和。Swin 没有抛弃 Self-Attention，只是限制了单层 Attention 的空间范围。

### 与 Transformer Encoder 的连接

Swin Block 仍然包含 Attention、FFN、残差连接和 LayerNorm。已经学过的 Transformer 基础仍然有效。

### 与 ViT 的连接

两者都会把图片区域表示成 tokens。ViT 通常保持平坦的 token 序列，Swin 则保留 H x W 网格并构建多个 Stage。

### 与 CNN 的连接

Window Attention 提供局部计算，Patch Merging 提供下采样，多个 Stage 提供层级特征。因此 Swin 同时具有 Transformer 的动态关系建模和 CNN 的视觉层级结构。

## 11. 本节小结

这一课需要真正记住四个结论：

1. Window Attention 把全局计算限制到局部窗口，降低高分辨率图片的注意力开销。
2. 固定窗口会阻断窗口间通信，Shifted Window 通过改变下一层的窗口分组解决这个问题。
3. Patch Merging 让高和宽减半、通道增加，形成层级特征。
4. Swin 仍然是 Transformer，只是加入了更适合图像的局部性和多尺度结构。

最重要的 shape 主线是：

$$
224\times224\times3
\rightarrow56\times56\times96
\rightarrow28\times28\times192
\rightarrow14\times14\times384
\rightarrow7\times7\times768
$$

下一课再单独学习 Window Attention 的窗口划分和窗口内 Q、K、V 形状，不在本课提前展开。

## 12. 自测问题

1. 普通 ViT 的全局 Self-Attention 为什么不利于处理高分辨率图片？
2. Window Attention 限制了什么，又保留了什么？
3. $56\times56$ 的特征图使用 $7\times7$ 窗口时，一共有多少个窗口？
4. 为什么固定窗口会阻碍不同区域的信息交流？
5. Shifted Window 移动的是图片、token，还是窗口分组方式？
6. Patch Merging 后，高、宽、token 数量和通道数分别怎样变化？
7. 为什么说 Swin 具有类似 CNN 的层级结构？
8. Swin 和普通 ViT 都保留了哪些 Transformer 组件？
9. Swin-T 从 Stage 1 到 Stage 4 的分辨率和通道数怎样变化？
10. Swin 为什么不等于 CNN？

### 自测参考答案

1. token 数量增加时，全局注意力矩阵按 token 数量的平方增长。
2. 它限制单层 Attention 的空间范围，但保留窗口内基于 Q、K、V 的动态关系建模。
3. 每边有 $56/7=8$ 个窗口，共有 $8\times8=64$ 个窗口。
4. 每层都按同一边界分组时，分属不同窗口的 tokens 无法直接进入同一次 Attention。
5. 移动的是窗口分组方式。
6. 高和宽各减半，token 数量变为四分之一，通道数通常变为两倍。
7. 它逐 Stage 降低空间分辨率、增加通道，并让深层 token 表示更大的区域。
8. Attention、FFN、残差连接和 LayerNorm。
9. $56\times56\times96$，$28\times28\times192$，$14\times14\times384$，$7\times7\times768$。
10. Swin 的局部关系仍通过 Attention 动态计算，而不是使用卷积核完成固定形式的局部加权。